# 10 — Vlastiti grounding nad PDF-om

**Četvrtak, 15:00.**

U 14h smo vidjeli da model odgovara **iz težina** — iz onoga što je zapamtio.
Sad mu dajemo *naš* dokument i tražimo da odgovara iz njega.

To je **RAG** (retrieval-augmented generation), i to je stvar koja će vam u
firmi trebati. Radimo u dvije razine:

1. **Cijeli dokument u kontekst.** Pet linija, radi odmah.
2. **Retrieval.** Uvodimo ga *tek kad razina 1 pukne* — i pukne zbog
   konkretnog ograničenja, ne zato što tako piše u knjizi.

Na kraju radimo test koji je najvažniji: **pitanje na koje u dokumentu
odgovora nema.**

In [ ]:
# --- SETUP: pokreni ovo prvo ---  [lares-setup-v1]
# Radi i u Colabu i lokalno. Sigurno je pokrenuti vise puta.
import os, sys, subprocess
REPO = "ai_bootcamp_foundations"
if "google.colab" in sys.modules:
    if not os.path.isdir(f"/content/{REPO}"):
        subprocess.run(["git", "clone", "-q",
                        f"https://github.com/hrvojenovak/{REPO}.git"],
                       cwd="/content", check=True)
    os.chdir(f"/content/{REPO}/notebooks")
print("cwd:", os.getcwd(), "| data ok:", os.path.isdir("../data"))

In [ ]:
%pip install -q pypdf

import json, time, random, requests, re, glob

# kljuc - isto kao u notebooku 9
API_KEY = None
if "google.colab" in sys.modules:
    from google.colab import userdata
    try:
        API_KEY = userdata.get("GOOGLE_API_KEY")
    except Exception as e:
        print("Secret nije dostupan:", type(e).__name__)
else:
    API_KEY = os.environ.get("GOOGLE_API_KEY")

MODEL = "gemini-2.5-flash"
print("kljuc:", "OK" if API_KEY else "NEMA — vidi notebook 9, korak 1")

## 1. Dokument

Radimo nad `data/pdf/izvjestaj_2025.pdf` — primjerom godišnjeg izvještaja
fiktivnog operatora prijenosnog sustava. **Svi podaci u njemu su izmišljeni**,
i to je za vježbu prednost: nijedan model ga nije mogao zapamtiti, pa točan
odgovor može doći samo iz dokumenta.

Ako radite sa svojim dokumentom, promijenite `DOC_PATH` ispod.

In [ ]:
DOC_PATH = "../data/reports/izvjestaj_2025.pdf"

if not os.path.exists(DOC_PATH):
    print(f"NEMA DOKUMENTA: {DOC_PATH}")
    print("Provjeri putanju, ili stavi svoj PDF i promijeni DOC_PATH.")
    print("\nPDF-ovi koje vidim u repou:")
    found = glob.glob("../data/**/*.pdf", recursive=True)
    print("\n".join(f"  {f}" for f in found) if found else "  (nijedan)")
else:
    print(f"{DOC_PATH}  ({os.path.getsize(DOC_PATH)/1024:.0f} kB)")

## 2. Ekstrakcija teksta

Jedna provjera koja spašava vježbu: **skenirani PDF-ovi ne rade.** `pypdf`
vrati prazan string, model dobije ništa, i ispada da je model glup. Zato
odmah mjerimo koliko je teksta izvučeno.

In [ ]:
from pypdf import PdfReader

def extract(path: str) -> str:
    return "\n".join(p.extract_text() or "" for p in PdfReader(path).pages)

doc = extract(DOC_PATH)
n_pages = len(PdfReader(DOC_PATH).pages)

print(f"{os.path.basename(DOC_PATH)}: {n_pages} stranica")
print(f"znakova: {len(doc)}  |  ~tokena: {len(doc)//4}  |  ~tokena/stranica: {len(doc)//4//n_pages}")

if len(doc) < 100 * n_pages:
    print("\n!!! PREMALO TEKSTA — PDF je vjerojatno SKENIRAN (slike, ne tekst).")
    print("    Treba OCR. Uzmite drugi dokument.")
else:
    print("\nekstrakcija OK")

## 3. Minimalni `ask()`

Isti kao u notebooku 9, skraćen. `thinking=False` jer Gemini 2.5 ima
razmišljanje uključeno po defaultu, a ti tokeni troše isti `maxOutputTokens`
budžet — bez toga vam se odgovor odreže na pola rečenice.

In [ ]:
STATS = {"requests": 0, "input_tokens": 0, "output_tokens": 0}

def ask(prompt: str, max_tokens: int = 2048) -> str:
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"
    body = {"contents": [{"role": "user", "parts": [{"text": prompt}]}],
            "generationConfig": {"temperature": 0.0, "maxOutputTokens": max_tokens,
                                 "thinkingConfig": {"thinkingBudget": 0}}}
    for attempt in range(1, 6):
        r = requests.post(url, params={"key": API_KEY}, json=body, timeout=90)
        if r.status_code in (429, 500, 502, 503, 504):
            if attempt == 5:
                return f"[odustajem: HTTP {r.status_code}]"
            d = min(32, 2 ** (attempt - 1)) * (0.5 + random.random())
            print(f"  HTTP {r.status_code}, cekam {d:.1f}s"); time.sleep(d); continue
        if r.status_code >= 400:
            return f"[HTTP {r.status_code}] {r.text[:200]}"
        j = r.json(); u = j.get("usageMetadata", {})
        STATS["requests"] += 1
        STATS["input_tokens"] += u.get("promptTokenCount", 0)
        STATS["output_tokens"] += u.get("candidatesTokenCount", 0)
        c = (j.get("candidates") or [{}])[0]
        txt = "".join(x.get("text", "") for x in (c.get("content") or {}).get("parts", []))
        if c.get("finishReason") == "MAX_TOKENS":
            txt += "\n[ODREZANO — povecaj max_tokens]"
        return txt.strip() or "[prazan odgovor]"
    return "[neocekivano]"

print(ask("Odgovori jednom rijecju: radis li?"))

## 4. Razina 1 — cijeli dokument u kontekst

Ovo je cijeli grounding. Nema vektorske baze, nema frameworka.

> **ZAMIJENITE `PITANJE`.** Uzmite brojku iz tablice na sredini svog
> dokumenta — nešto s decimalom. Opća pitanja ("o čemu govori izvještaj") ne
> rade, jer to model odgovori dovoljno uvjerljivo iz naslova pa razliku ne
> vidite.

In [ ]:
# Tocan odgovor je u Tablici 2.1 na 3. stranici: 1 347,8 MW
PITANJE = "Kolika je instalirana snaga vjetroelektrana na dan 31.12.2025.?"

# Druga pitanja za probu (odgovori su u razlicitim poglavljima):
#   "Koliki su bili gubici u prijenosu u GWh?"              -> 236,4   (pogl. 4)
#   "Kolika je raspolozivost vodova 220 kV?"                -> 98,87 % (pogl. 5)
#   "Koja je vrijednost projekta TS 400/110 kV Primjer Jug?" -> 41,8    (pogl. 7)

print("=== BEZ dokumenta (iz tezina) ===")
print(ask(PITANJE))

print("\n=== S dokumentom u kontekstu ===")
prompt = f"""Odgovori na pitanje koristeci ISKLJUCIVO tekst ispod.

--- DOKUMENT ---
{doc}
--- KRAJ ---

Pitanje: {PITANJE}"""
print(ask(prompt))

print("\n", STATS)

## 5. Zašto to ne skalira

Pogledajte `input_tokens` gore. Cijeli dokument ide u **svakom** pozivu.

Ovaj primjer je kratak i udobno stane — budimo pošteni oko toga. Ali stvarni
godišnji izvještaj ima 150–250 stranica, a korpus ih ima desetke.

Free tier ima ograničenje od ~250.000 tokena u minuti. **To je granica koja
prisili retrieval**, i to je jedini razlog zašto retrieval postoji. Ako vam
dokument stane, pošaljite ga cijelog — bit će i točnije.

In [ ]:
TPM = 250_000
tok_doc = len(doc) // 4
tok_str = tok_doc / max(1, n_pages)          # tokena po stranici, izmjereno

def fits(tok, label):
    print(f"  {label:38s} {tok:>9,.0f} tok  ->  {'NE STANE' if tok > TPM else 'stane'}")

print(f"limit po minuti (free tier): {TPM:,} tokena")
print(f"izmjereno: {tok_str:.0f} tokena po stranici\n")
fits(tok_doc, "nas primjer (10 str)")
fits(tok_doc * 30, "30 takvih primjera")
fits(200 * tok_str, "JEDAN stvarni izvjestaj (200 str)")
fits(200 * tok_str * 30, "korpus od 30 stvarnih izvjestaja")
print(f"\n-> granica je oko {TPM/tok_str:,.0f} stranica po minuti.")

## 6. Razina 2 — chunkanje i retrieval

Umjesto cijelog dokumenta, šaljemo samo dijelove koji su relevantni.

Retrieval je ovdje **naivan**: brojimo poklapanja riječi. To je namjerno —
želim da vidite da RAG u srži nije magija nego *pretraživanje*. Pravi sustavi
koriste embeddinge i to je bolje, ali ideja je ista.

`overlap` je tu da odgovor koji leži na granici dva chunka ne bude presječen.

In [ ]:
def chunk(text: str, size: int = 1200, overlap: int = 200) -> list[str]:
    out, i = [], 0
    while i < len(text):
        out.append(text[i:i+size]); i += size - overlap
    return out


def retrieve(query: str, chunks: list[str], k: int = 3) -> list[tuple]:
    words = [w.lower() for w in re.findall(r"\w{4,}", query)]
    scored = [(sum(c.lower().count(w) for w in words), i, c) for i, c in enumerate(chunks)]
    return sorted(scored, key=lambda t: -t[0])[:k]


chunks = chunk(doc)
hits = retrieve(PITANJE, chunks)
context = "\n---\n".join(c for _, _, c in hits)

print(f"chunkova: {len(chunks)}  |  odabrano: {len(hits)}")
for score, i, _ in hits:
    print(f"  chunk #{i:<3d} score={score}")
print(f"\nkontekst: {len(context)//4} tokena  (cijeli dokument: {len(doc)//4})")
print(f"usteda: {len(doc)//max(1,len(context))}x\n")

print("=== Odgovor iz retrievanog konteksta ===")
print(ask(f"""Odgovori koristeci ISKLJUCIVO tekst ispod.
Ako odgovora nema u tekstu, reci "Nije u dokumentu".

--- KONTEKST ---
{context}
--- KRAJ ---

Pitanje: {PITANJE}"""))

## 7. Najvažniji test: pitanje kojeg u dokumentu nema

Ovo je test koji dijeli one koji razumiju RAG od onih koji misle da ga
razumiju.

Postavite pitanje na koje u dokumentu **nema** odgovora. Model ga u dobrom
dijelu slučajeva **izmisli iz težina** — i to bez ikakve oznake da je to
učinio.

Zatim isto pitanje s izričitom instrukcijom da smije reći da ne zna.
Pokrenite više puta: **poslušnost nije 100%.**

In [ ]:
# U izvjestaju NEMA nicega o zaposlenima - zvuci kao da bi trebalo biti, a nije.
NEMA_ODGOVORA = "Koliki je bio prosjecni broj zaposlenih u 2025.?"

print("=== bez zastite ===")
print(ask(f"Odgovori na temelju teksta.\n\n{context}\n\nPitanje: {NEMA_ODGOVORA}"))

print("\n=== sa zastitom u promptu ===")
print(ask(f"""Odgovori koristeci ISKLJUCIVO tekst ispod. Ako odgovora nema u
tekstu, odgovori tocno: "Nije u dokumentu." Ne nagadaj.

--- KONTEKST ---
{context}
--- KRAJ ---

Pitanje: {NEMA_ODGOVORA}"""))

print("\n", STATS)

## 8. Za zapamtiti

**Grounding je davanje konteksta, ništa više.** Nema magije, nema
obaveznog frameworka. Sve što ste vidjeli je string konkatenacija.

**Retrieval nije zbog kvalitete, nego zbog veličine.** Ako vam dokument
stane u kontekst, pošaljite ga cijelog — bit će i točnije.

**Prompt zaštita pomaže, ali ne garantira.** Model i s instrukcijom "reci da
ne znaš" ponekad izmisli. Ako trebate pouzdanost, tražite **citat iz
konteksta** i provjerite ga programski.

**Skenirani PDF-ovi trebaju OCR.** Provjerite `len(text)` prije nego išta
drugo.

**Naivni keyword retrieval promašuje sinonime.** Ako pitate "vjetar" a u
dokumentu piše "vjetroelektrane", ovaj `retrieve()` to neće naći. To rješavaju
embeddinzi — Gemini ima i endpoint za njih, i to je logičan sljedeći korak.

---

U 16h agent dobiva `run_python` i sam odlučuje što će pozvati. Ovdje ste
*vi* odlučili što ide u kontekst. **To je razlika između workflowa i
agenta.**